### Library

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if project_root not in sys.path:
    sys.path.append(project_root)
print(f"Project root added: {project_root}")

In [ ]:
import multiprocessing as mp
if mp.get_start_method(allow_none=True) is None:
    mp.set_start_method('spawn')

import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', message='.*input_shape.*')
warnings.filterwarnings('ignore', message='.*structure of.*inputs.*')

import os, time, gc
from types import SimpleNamespace

import numpy as np
import pandas as pd
import time as time_module
from scipy.stats import t
from scipy.special import kv, gamma

import jax, jax.numpy as jnp

import tensorflow as tf
from tensorflow.keras import mixed_precision
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber
from tensorflow.keras import backend as K

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import OneHotEncoder

import optuna
import plotly.io as pio

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import random

### Environment setting

In [ ]:
np_f32 = np.float32
jnp_f32 = jnp.float32
dtype_basis = np.float32

jax.config.update("jax_enable_x64", False)

pio.renderers.default = "notebook"
warnings.filterwarnings("ignore", category=UserWarning)

os.environ.update({"TF_CPP_MIN_LOG_LEVEL": "2"})
optuna.logging.set_verbosity(optuna.logging.WARNING)

os.environ.setdefault("OMP_NUM_THREADS", "12")
os.environ.setdefault("MKL_NUM_THREADS", "12")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "12")

def init_hardware(dtype: str = "float32"):
    gpus = tf.config.list_physical_devices("GPU")
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    strategy = (tf.distribute.MirroredStrategy() if len(gpus) > 1 else tf.distribute.get_strategy())
    mixed_precision.set_global_policy(dtype)
    return strategy

strategy = init_hardware(dtype="float32")

### Auto save notebook

In [ ]:
from IPython.display import display, Javascript

def save_notebook():
    display(Javascript('IPython.notebook.save_checkpoint()'))
    current_time = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
    print(f"💾 Notebook saved at {current_time}")

### Our function

In [ ]:
from spherical_deepkriging.basis_functions.wendland.wenland import wendland
from spherical_deepkriging.basis_functions.mrts.mrts import mrts0

from spherical_deepkriging.models.deep_kriging import DeepKrigingTrainer, DeepKrigingDefaultTrainer
from spherical_deepkriging.configs import DeepKrigingModelConfig, DeepKrigingDefaultConfig
from spherical_deepkriging.models.universal_kriging import UniversalKriging

from rpy2.robjects.conversion import localconverter
from spherical_deepkriging.basis_functions.mrts_sphere.sphere import mrts_sphere, numpy2ri_converter

### Simulation Helper

In [ ]:
import numpy as np
import pandas as pd
import gc

def simulate_data(num_sample, outlier_ratio, outlier_multiplier, seed, scenario='E1'):
    """
    Non-GP: deterministic macro-trend + nonstationary anomalies + outlier injection.
    No noise, no Gaussian Process / Gamma noise component.
    """
    rng = np.random.default_rng(seed)

    phi   = rng.uniform(0, 2 * np.pi, num_sample)
    theta = np.arccos(rng.uniform(-1, 1, num_sample))
    lat_rad = np.pi/2 - theta
    lon_rad = phi - np.pi

    x_c = np.cos(lat_rad) * np.cos(lon_rad)
    y_c = np.cos(lat_rad) * np.sin(lon_rad)
    z_c = np.sin(lat_rad)

    base_wind        = 5.0
    westerlies_north = 18.0 * np.exp(-((lat_rad - np.pi/4)**2) / 0.05)
    westerlies_south = 22.0 * np.exp(-((lat_rad + np.pi/4)**2) / 0.04)
    doldrums         = -4.0 * np.exp(-(lat_rad**2) / 0.01)
    mountain_block   = np.where((lon_rad > 0.0) & (lon_rad < 1.0) &
                                (lat_rad > 0.1) & (lat_rad < 1.0), -12.0, 0.0)
    mean_trend = base_wind + westerlies_north + westerlies_south + doldrums + mountain_block

    num_anomalies = 60
    anomaly_lats  = np.arcsin(rng.uniform(-1, 1, num_anomalies))
    anomaly_lons  = rng.uniform(-np.pi, np.pi, num_anomalies)
    ax = np.cos(anomaly_lats) * np.cos(anomaly_lons)
    ay = np.cos(anomaly_lats) * np.sin(anomaly_lons)
    az = np.sin(anomaly_lats)

    anomaly_effect = np.zeros(num_sample)
    for i in range(num_anomalies):
        sq_dists = (x_c - ax[i])**2 + (y_c - ay[i])**2 + (z_c - az[i])**2
        amplitude = rng.uniform(-10.0, 18.0)
        radius    = rng.uniform(0.005, 0.03)
        anomaly_effect += amplitude * np.exp(-sq_dists / radius)

    mean_trend += anomaly_effect
    mean_trend   = np.maximum(mean_trend, 0.5)

    # No noise — pure deterministic trend
    z_final = mean_trend.copy().astype(np.float32)

    idx_outliers = rng.choice(num_sample, size=int(num_sample * outlier_ratio), replace=False)
    z_final[idx_outliers] *= outlier_multiplier

    print(f"\n=== Scenario {scenario} (Non-GP: Trend only + Outliers {outlier_ratio*100:.1f}% x{outlier_multiplier}) ===")
    print(f"Simulate Data | z mean: {np.mean(z_final):.4f} (\u00b1{np.std(z_final)/np.sqrt(num_sample):.4f}), "
          f"Variance: {np.var(z_final, ddof=0):.4f}, Range: [{np.min(z_final):.4f}, {np.max(z_final):.4f}]")

    lat_deg = np.rad2deg(lat_rad).astype(np.float32)
    lon_deg = np.rad2deg(lon_rad).astype(np.float32)

    del x_c, y_c, z_c, mean_trend, westerlies_north, westerlies_south, doldrums, mountain_block, anomaly_effect
    gc.collect()

    return pd.DataFrame({"longitude": lon_deg, "latitude": lat_deg, "z": z_final})


### Helper

In [ ]:
def data_preprocessing(data_frame):
    data = data_frame.copy()

    numeric_cols = ["longitude", "latitude", "z"]
    data[numeric_cols] = data[numeric_cols].apply(pd.to_numeric, errors="coerce")
    data.dropna(subset=numeric_cols, inplace=True)

    lon, lat = data["longitude"].to_numpy(), data["latitude"].to_numpy()

    norm_lon = (lon - lon.min()) / (lon.max() - lon.min())
    norm_lat = (lat - lat.min()) / (lat.max() - lat.min())

    location_data = np.column_stack([lat, lon]).astype("float32")
    location_data_norm = np.column_stack([norm_lon, norm_lat]).astype("float32")
    y_combined = data['z'].to_numpy().astype("float32")[:, None]

    # Handle
    categorical_data = None

    return location_data, location_data_norm, categorical_data, y_combined


def precompute_wendland(location, num_basis):
    parts = []
    for nb in num_basis:
        grid = np.column_stack(np.meshgrid(
            np.linspace(0, 1, int(np.sqrt(nb)), dtype=np_f32),
            np.linspace(0, 1, int(np.sqrt(nb)), dtype=np_f32),
        )).reshape(-1, 2).astype(np_f32)

        theta = np_f32(2.5)/np_f32(np.sqrt(nb))
        parts.append(
            wendland(location, grid, theta=theta, k=2)
        )

        # Clean up the memory
        del grid
        gc.collect()

    return np.hstack(parts).astype(dtype_basis, copy=False)


def precompute_max_mrts(distance_type, location_data, knot_num, order_max, knot=None):
    if knot is None:
        idx_knot = np.random.choice(location_data.shape[0], knot_num, replace=False)
        knot = location_data[idx_knot].astype(np_f32)
    else:
        idx_knot = None

    if distance_type == "sphere":
        with localconverter(numpy2ri_converter):
            res_r = mrts_sphere(knot, order_max, location_data.astype(np_f32))
        res_dict = dict(zip(res_r.names(), res_r))
        phi = np.asarray(res_dict["mrts"], dtype=dtype_basis)
    else:
        phi = np.asarray(
            mrts0(jnp.asarray(knot, dtype=jnp_f32), k=order_max, 
                  x=jnp.asarray(location_data, dtype=jnp_f32)), dtype=dtype_basis
        )

    return phi, idx_knot, knot


def prepare_data(categorical_data, basis, y_combined, seed, split_ratio=(0.8, 0.1, 0.1)):
    idx_all = np.arange(basis.shape[0])
    train_ratio, val_ratio, test_ratio = split_ratio
    
    train_val_x1, test_x1 = train_test_split(
        idx_all, train_size=train_ratio+val_ratio, random_state=seed)
    train_x1, val_x1 = train_test_split(
        train_val_x1, train_size=train_ratio/(train_ratio+val_ratio), random_state=seed)
    
    X_train_cont, X_val_cont, X_test_cont = (
        basis[train_x1], basis[val_x1], basis[test_x1])
    y_train, y_val, y_test = (
        y_combined[train_x1], y_combined[val_x1], y_combined[test_x1])
    
    def flatten(targets):
        return targets.reshape(-1).astype(np_f32, copy=False)
    y_train_flat, y_val_flat, y_test_flat = map(flatten, [y_train, y_val, y_test])

    def flatten(covariates):
        return covariates.reshape(-1, basis.shape[1]).astype(np_f32)
    X_train_cont_flat, X_val_cont_flat, X_test_cont_flat = map(flatten, [X_train_cont, X_val_cont, X_test_cont])
    
    # Handle categorical features
    if categorical_data is None:
        zero_vector = lambda n: np.zeros((n, 0), dtype=np_f32)
        X_train_cat, X_val_cat, X_test_cat = map(zero_vector, [len(X_train_cont_flat), len(X_val_cont_flat), len(X_test_cont_flat)])
    else:
        cat_train = categorical_data.reshape(-1, 1)[train_x1]
        cat_val = categorical_data.reshape(-1, 1)[val_x1]
        cat_test = categorical_data.reshape(-1, 1)[test_x1]
        
        OHE = OneHotEncoder(sparse_output=False, dtype=np_f32)
        X_train_cat = OHE.fit_transform(cat_train).astype(np_f32)
        X_val_cat = OHE.transform(cat_val).astype(np_f32)
        X_test_cat = OHE.transform(cat_test).astype(np_f32)
    
    return (X_train_cont_flat, X_train_cat, y_train_flat,
            X_val_cont_flat, X_val_cat, y_val_flat,
            X_test_cont_flat, X_test_cat, y_test_flat)


def train_eval(name_model, epochs, batch_size, loss, dropout_rate,
               X_train, X_train_cat, y_train,
               X_val, X_val_cat, y_val,
               X_test, X_test_cat, y_test):

    # 1. Force deterministic weights for this specific seed
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    if name_model in ["OLS_wendland", "OLS_sphere"]:
        t0 = time.time()
        model = LinearRegression().fit(X_train, y_train)
            
        val_loss = float(mean_squared_error(y_val, model.predict(X_val)))
        y_pred = model.predict(X_test).astype(np_f32).reshape(-1)
        training_time = time.time() - t0
        
        metrics = {
            "Model": name_model,
            "Val_loss": float(val_loss),
            "MSPE": float(mean_squared_error(y_test, y_pred)),
            "RMSE": float(np.sqrt(float(mean_squared_error(y_test, y_pred)))),
            "MAE": float(mean_absolute_error(y_test, y_pred)),
            "R2": float(r2_score(y_test, y_pred)),
            "Time": float(training_time),
        }
        
        return metrics, model
    
    elif name_model == "DeepKriging_wendland":
        config = DeepKrigingDefaultConfig(
            input_dim=X_train.shape[1],
            output_type='continuous',
            optimizer=Adam(learning_rate=1e-3),
            loss=loss,
            epochs=epochs,
            batch_size=batch_size,
            verbose=0
        )

    elif name_model == "DeepKriging_wendland*":
        optimizer = Adam(learning_rate=5e-3)
        config = DeepKrigingModelConfig(
            input_dim=X_train.shape[1],
            output_type='continuous',
            hidden_layers=[1024, 512, 256, 128, 64],
            activation='relu',
            dropout_rate=dropout_rate,
            optimizer=optimizer,
            loss=loss,
            metrics=['mae'],
            epochs=epochs,
            batch_size=batch_size,
            patience=40,
            verbose=0
        )

    elif name_model in ["DeepKriging_mrts", "DeepKriging_sphere", "DeepKriging_sphere_Huber"]:
        optimizer = Adam(learning_rate=5e-3)
        config = DeepKrigingModelConfig(
            input_dim=X_train.shape[1],
            output_type='continuous',
            hidden_layers=[1024, 512, 256, 128, 64],
            activation='relu',
            dropout_rate=dropout_rate,
            optimizer=optimizer,
            loss=loss,
            metrics=['mae'],
            epochs=epochs,
            batch_size=batch_size,
            patience=40,
            verbose=0
        )

    t0 = time.time()
    with strategy.scope():
        if name_model == "DeepKriging_wendland":
            model = DeepKrigingDefaultTrainer(config)
        elif name_model == "DeepKriging_wendland*":
            model = DeepKrigingTrainer(config)
        else:
            model = DeepKrigingTrainer(config)
        model.model.compile(optimizer=config.optimizer, loss=config.loss, metrics=config.metrics)

    checkpoint_path = f"best_{name_model}_{time.time_ns()}.weights.h5"
    if name_model == "DeepKriging_wendland":
        callbacks = [
            tf.keras.callbacks.ModelCheckpoint(
                filepath=checkpoint_path, monitor="val_loss", mode="min",
                save_best_only=True, save_weights_only=True, verbose=0)
        ]
    else:
        callbacks = [
            tf.keras.callbacks.ModelCheckpoint(
                filepath=checkpoint_path, monitor="val_loss", mode="min",
                save_best_only=True, save_weights_only=True, verbose=0),
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=config.patience, restore_best_weights=True, verbose=0),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5, patience=max(5, config.patience // 2), min_lr=1e-6, verbose=0)
        ]

    train_dataset = tf.data.Dataset.from_tensor_slices((
        (X_train, X_train_cat), y_train
    )).batch(config.batch_size).prefetch(tf.data.AUTOTUNE)

    val_dataset = tf.data.Dataset.from_tensor_slices((
        (X_val, X_val_cat), y_val
    )).batch(config.batch_size).prefetch(tf.data.AUTOTUNE)

    history = model.model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=epochs,
        callbacks=callbacks,
        verbose=0
    )

    if os.path.exists(checkpoint_path):
        model.model.load_weights(checkpoint_path)
        os.remove(checkpoint_path)
    
    val_loss = float(np.min(history.history["val_loss"]))
    y_pred = model.model.predict([X_test, X_test_cat], verbose=0).reshape(-1).astype(np_f32)
    training_time = time.time() - t0

    metrics = {
        "Model": name_model,
        "Val_loss": float(val_loss),
        "MSPE": float(mean_squared_error(y_test, y_pred)),
        "RMSE": float(np.sqrt(float(mean_squared_error(y_test, y_pred)))),
        "MAE": float(mean_absolute_error(y_test, y_pred)),
        "R2": float(r2_score(y_test, y_pred)),
        "Time": float(training_time),
    }
    
    del train_dataset, val_dataset
    gc.collect()
    
    return metrics, model


def cleanup_tf_session():
    K.clear_session()
    gc.collect()
    try:
        tf.keras.backend.clear_session()
    except:
        pass

In [ ]:
def plot_robinson(ax, longitude, latitude, value, vmin, vmax, title):
    """Plot data on Robinson projection"""
    ax.set_global()
    ax.add_feature(cfeature.LAND, facecolor="white", alpha=0.3)
    ax.add_feature(cfeature.OCEAN, facecolor="white", alpha=0.3)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.3, alpha=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.2, alpha=0.5)

    sc = ax.scatter(longitude, latitude, c=value, 
                    cmap=mcolors.LinearSegmentedColormap.from_list("teal-yellow-red", ["#00AAAA", "#FFFFBB", "#FF3333"], N=256), 
                    s=10, transform=ccrs.PlateCarree(), vmin=vmin, vmax=vmax)

    ax.set_title(title, fontsize=10, pad=3)
    return sc


def create_subplot_robinson(fig, position, locations, values, vmin, vmax, title, plot_type='prediction', cbar_label=None):
    """Create subplot with Robinson projection"""
    ax = fig.add_subplot(*position, projection=ccrs.Robinson())
    
    # Choose colormap based on plot type
    if plot_type == 'residual':
        cmap = mcolors.LinearSegmentedColormap.from_list("blue-white-red", ["#2166AC", "#F7F7F7", "#B2182B"], N=256)
    else:
        cmap = mcolors.LinearSegmentedColormap.from_list("teal-yellow-red", ["#00AAAA", "#FFFFBB", "#FF3333"], N=256)
    
    # Set global view
    ax.set_global()
    ax.add_feature(cfeature.LAND, facecolor="white", alpha=0.3)
    ax.add_feature(cfeature.OCEAN, facecolor="white", alpha=0.3)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.3, alpha=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.2, alpha=0.5)
    
    # Plot scatter
    sc = ax.scatter(locations['longitude'], locations['latitude'], c=values, 
                    cmap=cmap, s=10, transform=ccrs.PlateCarree(), 
                    vmin=vmin, vmax=vmax)
    
    ax.set_title(title, fontsize=10, pad=3)
    
    # Add colorbar
    cbar = plt.colorbar(sc, ax=ax, orientation='horizontal', fraction=0.046, pad=0.04, shrink=0.8)
    
    if cbar_label is None:
        cbar_label = "Residual" if plot_type == 'residual' else "Prediction Value"
    
    # Increased fontsize to match old version
    cbar.set_label(cbar_label, fontsize=10)
    cbar.ax.tick_params(labelsize=7)
    
    return ax, sc


def visualize_comparison(dataframe, models_dict, basis_dict, y_combined, seed, model_list=None, experiment_info=None):
    """Visualize model predictions and residuals using Robinson projection"""
    if model_list is None:
        model_list = ['DeepKriging_sphere', 'DeepKriging_sphere_Huber', 'UniversalKriging']
    
    idx_all = np.arange(len(y_combined))
    train_val_idx, test_idx = train_test_split(idx_all, train_size=0.9, random_state=seed)
    y_test = y_combined[test_idx].reshape(-1)
    test_locations = dataframe.iloc[test_idx]
    
    predictions = {}
    for model_name in model_list:
        if model_name not in models_dict or models_dict[model_name] is None:
            continue
        
        model = models_dict[model_name]
        X_test = basis_dict[model_name][test_idx]
        
        if "DeepKriging" in model_name:
            X_test_cat = np.zeros((len(X_test), 0), dtype=np.float32)
            y_pred = model.model.predict([X_test, X_test_cat], verbose=0).reshape(-1)
        elif model_name == "UniversalKriging":
            coords_test = dataframe[['longitude', 'latitude']].iloc[test_idx].values.astype(np.float32)
            y_pred = model.predict(coords_test, X_test, return_centered=False)
        else:
            y_pred = model.predict(X_test).reshape(-1)
        
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        order = models_dict.get(f"{model_name}_order", "")
        
        predictions[model_name] = {
            'values': y_pred,
            'rmse': rmse,
            'order': order
        }
    
    # Calculate global min/max for consistent color scale
    all_values = [dataframe['z'].values] + [p['values'] for p in predictions.values()]
    all_values_concat = np.concatenate(all_values)
    vmin = np.percentile(all_values_concat, 2)
    vmax = np.percentile(all_values_concat, 98)
    
    # Figure 1: Predictions comparison
    fig1 = plt.figure(figsize=(16, 14))
    
    noise_info = ""
    if experiment_info:
        noise_info = f"Noise={experiment_info.get('noise', 'None')}"
        if experiment_info.get('noise_var'):
            noise_info += f", Var={experiment_info['noise_var']}"
    
    # Plot real data
    create_subplot_robinson(
        fig1, (2, 2, 1), dataframe, dataframe['z'], vmin, vmax,
        f'Real Data (n={len(dataframe)})',
        plot_type='prediction'
    )
    
    # Plot predictions
    subplot_positions = [(2, 2, 2), (2, 2, 3), (2, 2, 4)]
    for i, model_name in enumerate(model_list):
        if model_name in predictions:
            pred = predictions[model_name]
            display_name = model_name.replace('DeepKriging_sphere', 'DK_S').replace('_Huber', '_H').replace('UniversalKriging', 'UK')
            
            create_subplot_robinson(
                fig1, subplot_positions[i], test_locations, pred['values'], vmin, vmax,
                f"{display_name} (order={pred['order']}) | Test n={len(test_idx)} | RMSE={pred['rmse']:.4f}",
                plot_type='prediction'
            )
    
    # Modified title fontsize and layout to match old version
    plt.suptitle(
        "Prediction Comparison: Real Data vs. Models Predict\n"
        f"Stationary Gaussian processes + Eggholder (without noise and outliers)",
        fontsize=20, fontweight='bold', y=0.84
    )
    # Reverted to tight_layout with rect, as in the old version
    plt.tight_layout(rect=[0, 0.02, 1, 0.94])
    plt.show()
    plt.close(fig1)
    

    # Figure 2: Residuals comparison
    fig2 = plt.figure(figsize=(18, 6))
    
    residuals_map = {k: (y_test - predictions[k]['values']) 
                     for k in model_list if k in predictions}
    vmax_abs = max(np.max(np.abs(r)) for r in residuals_map.values())
    
    for i, model_name in enumerate(model_list):
        if model_name in predictions:
            residuals = residuals_map[model_name]
            display_name = model_name.replace('DeepKriging_sphere', 'DK_S').replace('_Huber', '_H').replace('UniversalKriging', 'UK')
            
            create_subplot_robinson(
                fig2, (1, 3, i+1), test_locations, residuals, -vmax_abs, vmax_abs,
                f"{display_name} Residuals (order={predictions[model_name]['order']})",
                plot_type='residual'
            )
    
    # Modified title fontsize and layout to match old version
    plt.suptitle(
        f"Residuals Comparison | Stationary Gaussian processes + Eggholder (without noise and outliers)",
        fontsize=20, fontweight='bold', y=0.84
    )
    # Reverted to tight_layout with rect
    plt.tight_layout(rect=[0, 0.02, 1, 0.94])
    plt.show()
    plt.close(fig2)
    
    return predictions, test_idx

### Experiment Setup

In [ ]:
# Model Setup
seed             = 50
OUTLIER_RATIO    = 0.025
OUTLIER_MULTIPLIER = 5
epochs           = 500
batch_size       = 256
num_sample       = 2500
huber_delta      = 1.345

# Basis Setup
num_basis  = [10**2, 19**2, 37**2]
knot_num   = 1400
order_max  = 1400
base_orders = [10, 50, 100, 150, 200, 1000]

repeat_experiment = 50

print(f"noGP outliers experiment: {repeat_experiment} repeats, seed={seed}")
print(f"Outliers: {OUTLIER_RATIO*100:.1f}% x{OUTLIER_MULTIPLIER}")


In [ ]:
import json, subprocess, sys

CHECKPOINT_PATH = "checkpoint_noGP_outliers.json"
SCRIPT_PATH     = os.path.join(os.getcwd(), "run_single_repeat_noGP_outliers.py")
PYTHON_EXE      = sys.executable

# ── load checkpoint ────────────────────────────────────────────────────────────
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        ckpt = json.load(f)
    experiment_results = ckpt["experiment_results"]
    completed_repeats  = set(ckpt["completed_repeats"])
    print(f"▶ Resuming: {len(completed_repeats)}/{repeat_experiment} repeats already done.")
else:
    experiment_results = {
        m: {"MSPE": [], "RMSE": [], "MAE": [], "R2": []}
        for m in ["OLS_wendland", "OLS_sphere", "DeepKriging_wendland", "DeepKriging_mrts",
                  "DeepKriging_sphere", "DeepKriging_sphere_Huber", "UniversalKriging"]
    }
    completed_repeats = set()
    print("▶ Starting fresh.")

# ── main loop ──────────────────────────────────────────────────────────────────
for repeat in range(repeat_experiment):

    if repeat in completed_repeats:
        print(f"⏭  Skip repeat {repeat+1}/{repeat_experiment} (checkpoint)")
        continue

    print(f"\n{'='*70}")
    print(f"🏃 Repeat {repeat+1}/{repeat_experiment}  seed={seed+repeat}")
    print(f"{'='*70}")

    out_json = f"repeat_{repeat}_noGP_outliers_results.json"

    try:
        result = subprocess.run(
            [PYTHON_EXE, SCRIPT_PATH, str(repeat), str(seed), out_json],
            capture_output=False,
            check=True,
            timeout=7200,
        )
    except subprocess.CalledProcessError as e:
        print(f"❌ Repeat {repeat+1} subprocess exited with code {e.returncode}. Skipping.")
        continue
    except subprocess.TimeoutExpired:
        print(f"❌ Repeat {repeat+1} timed out. Skipping.")
        continue
    except Exception as e:
        print(f"❌ Repeat {repeat+1} failed: {e}. Skipping.")
        continue

    if not os.path.exists(out_json):
        print(f"❌ No output JSON for repeat {repeat+1}. Skipping.")
        continue

    with open(out_json) as f:
        res = json.load(f)
    os.remove(out_json)

    best_orders = res["best_orders"]
    metrics_map = res["metrics"]

    table_rows = []
    for m in ["OLS_wendland", "OLS_sphere", "DeepKriging_wendland", "DeepKriging_mrts",
              "DeepKriging_sphere", "DeepKriging_sphere_Huber", "UniversalKriging"]:
        met = metrics_map[m]
        table_rows.append({
            "Model": m, "Param": best_orders.get(m, "--"),
            "MSPE": f"{met['MSPE']:.4f}", "RMSE": f"{met['RMSE']:.4f}",
            "MAE":  f"{met['MAE']:.4f}",  "R2":   f"{met['R2']:.4f}",
        })
    import pandas as _pd
    print("\n", _pd.DataFrame(table_rows).to_markdown(index=False, tablefmt="github"), sep="")

    for m in experiment_results:
        experiment_results[m]["MSPE"].append(metrics_map[m]["MSPE"])
        experiment_results[m]["RMSE"].append(metrics_map[m]["RMSE"])
        experiment_results[m]["MAE"].append(metrics_map[m]["MAE"])
        experiment_results[m]["R2"].append(metrics_map[m]["R2"])

    completed_repeats.add(repeat)
    with open(CHECKPOINT_PATH, "w") as f:
        json.dump({"experiment_results": experiment_results,
                   "completed_repeats": list(completed_repeats)}, f)

    print(f"✅ Repeat {repeat+1}/{repeat_experiment} done — checkpoint saved.")

print(f"\n🎉 All done: {len(completed_repeats)}/{repeat_experiment} repeats completed.")


### Summary of All Experiments

In [ ]:
import json, numpy as np, pandas as pd

MODELS = ["OLS_wendland", "OLS_sphere", "DeepKriging_wendland", "DeepKriging_mrts",
          "DeepKriging_sphere", "DeepKriging_sphere_Huber", "UniversalKriging"]

with open(CHECKPOINT_PATH) as f:
    ckpt = json.load(f)
results = ckpt["experiment_results"]
n = len(next(iter(results.values()))["MSPE"])

print("\n" + "="*80)
print(f"📊 Summary — {n} Repeats")
print("    Scenario: Non-GP + Outliers (2.5% × 5×), Nonstationary Trend")
print("="*80)

rows = []
for m in MODELS:
    vals = results[m]
    rows.append({
        "Model":           m,
        "N":               len(vals["MSPE"]),
        "MSPE (mean±std)": f"{np.mean(vals['MSPE']):.2f}±{np.std(vals['MSPE']):.2f}",
        "RMSE (mean±std)": f"{np.mean(vals['RMSE']):.2f}±{np.std(vals['RMSE']):.2f}",
        "MAE  (mean±std)": f"{np.mean(vals['MAE']):.2f}±{np.std(vals['MAE']):.2f}",
        "R²   (mean±std)": f"{np.mean(vals['R2']):.3f}±{np.std(vals['R2']):.3f}",
    })

df = pd.DataFrame(rows)
print("\n", df.to_markdown(index=False, tablefmt="github"), sep="")

best = min(rows, key=lambda r: float(r["RMSE (mean±std)"].split("±")[0]))
print(f"\n🏆 Best Model: {best['Model']}  RMSE={best['RMSE (mean±std)']}")

print("\n── Ranking by mean RMSE ──")
for i, r in enumerate(sorted(rows, key=lambda r: float(r["RMSE (mean±std)"].split("±")[0])), 1):
    print(f"  {i}. {r['Model']:<35} RMSE={r['RMSE (mean±std)']}")
